# NB1: Árvore de decisão (ID3)

**Projeto 02, FAI, G07**

> **Notebook em construção, bloco a bloco** (ver `docs/pendentes.md`, secção 3).
> Por agora tem só a implementação do algoritmo, testada no exemplo «Restaurant» do docente.
> As fases CRISP-DM 1-3 (Business Understanding, Data Understanding, Data Preparation) entram nos blocos N1-N2.
> A **EDA completa** fica no notebook de apoio `NB0_EDA.ipynb`; aqui vai só o resumo das conclusões (decisão D11).

## 4. Modeling: implementação do ID3 «à mão»

Pela orientação O5 não se usa nenhuma biblioteca de ML: cada peça do algoritmo é escrita de raiz e testada antes de passarmos à seguinte.

### 4.0 Bibliotecas

Só as bibliotecas que o docente usa (FAI33 p.8), agrupadas com um comentário por grupo, como nos slides.

In [1]:
# ======================================================================
# 4.0 BIBLIOTECAS
# O que faz: importa as únicas bibliotecas usadas no notebook (as do docente, FAI33 p.8).
# Não se importa nenhuma biblioteca de ML (orientação O5).
# ======================================================================

# linear algebra and data processing libraries
import numpy as np      # log2 para a fórmula da entropia
import pandas as pd     # tabela de exemplos (DataFrame)

# Utils
from collections import Counter   # contar quantos exemplos há de cada classe

### 4.1 Dados de teste: exemplo «Restaurant» (FAI32 p.17)

São 12 exemplos e 10 atributos. A classe é `WillWait`: esperamos ou não por uma mesa?
A tabela foi copiada do slide 17 tal como está. Serve para confirmar que o código dá os mesmos resultados que os slides (FAI32 p.18-19: a raiz da árvore deve ser *Patrons*).

| Atributo | Significado | Valores |
|---|---|---|
| `Alt` | Há um restaurante alternativo perto? | Yes / No |
| `Bar` | Tem bar confortável para esperar? | Yes / No |
| `Fri` | É sexta ou sábado? | Yes / No |
| `Hun` | Temos fome? | Yes / No |
| `Pat` | Quantas pessoas estão no restaurante (*Patrons*) | None / Some / Full |
| `Price` | Gama de preço | \$ / \$\$ / \$\$\$ |
| `Rain` | Está a chover? | Yes / No |
| `Res` | Fizemos reserva? | Yes / No |
| `Type` | Tipo de restaurante | French / Italian / Thai / Burger |
| `Est` | Tempo de espera estimado (minutos) | 0-10 / 10-30 / 30-60 / >60 |
| **`WillWait`** | **Classe: esperamos?** | Yes / No |

In [2]:
# ======================================================================
# 4.1 DADOS DE TESTE: RESTAURANT (FAI32 p.17)
# O que faz: cria a tabela `restaurant` com os 12 exemplos do slide 17.
# Para que serve: testar cada função do ID3 num caso pequeno cujo resultado já conhecemos.
# Resultado: DataFrame com 12 linhas (x1..x12), 10 atributos e a classe WillWait.
# ======================================================================

# Cada linha é um exemplo (x1 a x12) e cada coluna é um atributo; a última coluna é a classe.
colunas = ['Alt', 'Bar', 'Fri', 'Hun', 'Pat', 'Price', 'Rain', 'Res', 'Type', 'Est', 'WillWait']

exemplos = [
    # Alt    Bar    Fri    Hun    Pat     Price  Rain   Res    Type       Est      WillWait
    ['Yes', 'No',  'No',  'Yes', 'Some', '$$$', 'No',  'Yes', 'French',  '0-10',  'Yes'],  # x1
    ['Yes', 'No',  'No',  'Yes', 'Full', '$',   'No',  'No',  'Thai',    '30-60', 'No'],   # x2
    ['No',  'Yes', 'No',  'No',  'Some', '$',   'No',  'No',  'Burger',  '0-10',  'Yes'],  # x3
    ['Yes', 'No',  'Yes', 'Yes', 'Full', '$',   'Yes', 'No',  'Thai',    '10-30', 'Yes'],  # x4
    ['Yes', 'No',  'Yes', 'No',  'Full', '$$$', 'No',  'Yes', 'French',  '>60',   'No'],   # x5
    ['No',  'Yes', 'No',  'Yes', 'Some', '$$',  'Yes', 'Yes', 'Italian', '0-10',  'Yes'],  # x6
    ['No',  'Yes', 'No',  'No',  'None', '$',   'Yes', 'No',  'Burger',  '0-10',  'No'],   # x7
    ['No',  'No',  'No',  'Yes', 'Some', '$$',  'Yes', 'Yes', 'Thai',    '0-10',  'Yes'],  # x8
    ['No',  'Yes', 'Yes', 'No',  'Full', '$',   'Yes', 'No',  'Burger',  '>60',   'No'],   # x9
    ['Yes', 'Yes', 'Yes', 'Yes', 'Full', '$$$', 'No',  'Yes', 'Italian', '10-30', 'No'],   # x10
    ['No',  'No',  'No',  'No',  'None', '$',   'No',  'No',  'Thai',    '0-10',  'No'],   # x11
    ['Yes', 'Yes', 'Yes', 'Yes', 'Full', '$',   'No',  'No',  'Burger',  '30-60', 'Yes'],  # x12
]

# Construir a tabela e dar a cada linha o nome do exemplo (x1, x2, ...), como no slide
restaurant = pd.DataFrame(exemplos, columns=colunas, index=[f'x{i}' for i in range(1, 13)])

# Mostrar a tabela para comparar com o slide 17
restaurant

,Alt,Bar,Fri,Hun,Pat,Price,Rain,Res,Type,Est,WillWait
x1,Yes,No,No,Yes,Some,$$$,No,Yes,French,0-10,Yes
x2,Yes,No,No,Yes,Full,$,No,No,Thai,30-60,No
x3,No,Yes,No,No,Some,$,No,No,Burger,0-10,Yes
x4,Yes,No,Yes,Yes,Full,$,Yes,No,Thai,10-30,Yes
x5,Yes,No,Yes,No,Full,$$$,No,Yes,French,>60,No
x6,No,Yes,No,Yes,Some,$$,Yes,Yes,Italian,0-10,Yes
x7,No,Yes,No,No,None,$,Yes,No,Burger,0-10,No
x8,No,No,No,Yes,Some,$$,Yes,Yes,Thai,0-10,Yes
x9,No,Yes,Yes,No,Full,$,Yes,No,Burger,>60,No
x10,Yes,Yes,Yes,Yes,Full,$$$,No,Yes,Italian,10-30,No


### 4.2 Bloco A1: Entropia (FAI32 p.16)

A **entropia** mede a «desordem» de um conjunto de exemplos, ou seja, quanta informação falta para saber a classe de um exemplo qualquer. Se as classes estão nas proporções $p_1, p_2, \dots, p_m$:

$$E(p_1, p_2, \dots, p_m) = -\sum_{i=1}^{m} p_i \log_2(p_i)$$

Como se lê:
- **$E = 0$**: todos os exemplos são da mesma classe (nó **puro**). Não falta informação nenhuma.
- **$E = 1$** (com 2 classes): metade de cada classe. É a incerteza máxima.
- O ID3 vai usar a entropia para escolher, em cada nó, o atributo que **mais baixa a entropia** (o ganho de informação, bloco A3).

A função abaixo segue a fórmula termo a termo: conta os exemplos de cada classe, transforma as contagens em proporções $p_i$ e soma $-p_i \log_2(p_i)$.

In [3]:
# ======================================================================
# 4.2 BLOCO A1: FUNÇÃO entropia(y)
# O que faz: define a função que calcula a entropia de um conjunto de exemplos.
# Entrada: lista (ou coluna) com a classe de cada exemplo.
# Saída: número >= 0 (0 = conjunto puro; 1 = duas classes em partes iguais).
# Onde vai ser usada: no ganho de informação (bloco A3) para escolher o atributo de cada nó.
# ======================================================================

def entropia(y):
    """
    Entropia de um conjunto de exemplos (FAI32 p.16):
        E(p1, ..., pm) = - soma de i=1 a m de  p_i * log2(p_i)

    y : lista (ou coluna do DataFrame) com a classe de cada exemplo,
        por exemplo ['Yes', 'No', 'Yes', ...].
    Devolve um número >= 0: 0 se o conjunto é puro, 1 se tem 2 classes em partes iguais.
    """
    # Número total de exemplos no conjunto
    n = len(y)

    # Conjunto vazio: não há exemplos para classificar, por isso não há incerteza.
    # (Convenção para evitar dividir por zero quando um ramo da árvore fica sem exemplos.)
    if n == 0:
        return 0.0

    # Passo 1: contar quantos exemplos há de cada classe.
    # Ex.: Counter(['Yes', 'No', 'Yes']) dá {'Yes': 2, 'No': 1}
    contagens = Counter(y)

    # Passo 2 e 3: para cada classe, calcular p_i e somar o termo -p_i * log2(p_i).
    # Só aparecem no Counter as classes que existem (contagem > 0), por isso p_i nunca é 0
    # e nunca se calcula log2(0), que não está definido.
    e = 0.0
    for classe, contagem in contagens.items():
        p_i = contagem / n             # proporção de exemplos desta classe
        e -= p_i * np.log2(p_i)        # o sinal menos da fórmula: log2(p_i) <= 0, logo cada termo soma >= 0

    return float(e)

#### Teste do bloco A1 no «Restaurant»

Valores esperados, **calculados à mão** antes de correr o código:

| Conjunto | Exemplos | Classes | Conta | Esperado |
|---|---|---|---|---|
| Todos | x1-x12 | 6 Yes / 6 No | $-(\tfrac12\log_2\tfrac12 + \tfrac12\log_2\tfrac12)$ | **1,000** |
| `Pat` = None | x7, x11 | 0 Yes / 2 No | nó puro | **0,000** |
| `Pat` = Some | x1, x3, x6, x8 | 4 Yes / 0 No | nó puro | **0,000** |
| `Pat` = Full | x2, x4, x5, x9, x10, x12 | 2 Yes / 4 No | $-(\tfrac13\log_2\tfrac13 + \tfrac23\log_2\tfrac23) = 0{,}528 + 0{,}390$ | **0,918** |

In [4]:
# ======================================================================
# 4.2 BLOCO A1: TESTE DA ENTROPIA NO RESTAURANT
# O que faz: compara a entropia calculada pelo código com as contas feitas à mão
# (tabela acima) e imprime OK ou FALHOU em cada caso.
# Se todos derem OK, a função entropia está correta e podemos avançar para o bloco A2.
# ======================================================================

def verificar(descricao, obtido, esperado, tolerancia=0.001):
    """Compara o valor calculado pelo código com o valor calculado à mão e imprime OK ou FALHOU."""
    resultado = 'OK' if abs(obtido - esperado) < tolerancia else 'FALHOU'
    print(f"{resultado:7} {descricao:32} obtido = {obtido:.3f}   esperado = {esperado:.3f}")

# Coluna da classe (o que queremos prever)
classe = restaurant['WillWait']

# 1) Entropia do conjunto completo: 6 Yes / 6 No
verificar("Todos os 12 exemplos", entropia(classe), 1.000)

# 2) Entropia de cada ramo de Patrons: filtramos os exemplos com esse valor e medimos a entropia da classe
verificar("Pat = None  (x7, x11)", entropia(classe[restaurant['Pat'] == 'None']), 0.000)
verificar("Pat = Some  (x1, x3, x6, x8)", entropia(classe[restaurant['Pat'] == 'Some']), 0.000)
verificar("Pat = Full  (6 exemplos)", entropia(classe[restaurant['Pat'] == 'Full']), 0.918)

# 3) Casos-limite: conjunto vazio e conjunto com uma só classe
verificar("Conjunto vazio", entropia([]), 0.000)
verificar("Só uma classe", entropia(['Yes', 'Yes', 'Yes']), 0.000)

OK      Todos os 12 exemplos             obtido = 1.000   esperado = 1.000
OK      Pat = None  (x7, x11)            obtido = 0.000   esperado = 0.000
OK      Pat = Some  (x1, x3, x6, x8)     obtido = 0.000   esperado = 0.000
OK      Pat = Full  (6 exemplos)         obtido = 0.918   esperado = 0.918
OK      Conjunto vazio                   obtido = 0.000   esperado = 0.000
OK      Só uma classe                    obtido = 0.000   esperado = 0.000


**Leitura do resultado:** o conjunto completo tem entropia 1 (incerteza máxima). Depois de dividir por *Patrons*, os ramos None e Some ficam puros (entropia 0) e só o Full mantém incerteza (0,918). Isto mostra porque é que *Patrons* é um bom atributo para a raiz (FAI32 p.18). O bloco A3 vai pôr isto num número só: o **ganho de informação**.